In [ ]:
# Install if you haven"t already.
#%pip install -U -qq pyfabricops 

# Parameters 
project_name = "fish_trade" # Used for workspaces, pipelines and repository
capacity = "TrialBrazilSouth"      # Or name (see pf.list_capacities()) 

# Azure DevOps required parameters
# Create an Azure DevOps organization and project, and set the parameters below accordingly. 
# The script will create a repository in the specified project, but it needs to exist beforehand.
# You can use the same project for multiple environments, as the script will create different branches and suffixes for workspaces and pipelines.
github_username = "jaircampelo"
github_repository_name = "fish-trade-analytics-fabric" 
github_url = f"https://github.com/{github_username}/{github_repository_name}"

# Workspaces will be created with suffixes based on branches, e.g. main -> FabricOpsFlow-PRD, develop -> FabricOpsFlow-DEV
branches = [      
    {"branch": "develop", "suffix": "DEV"},
    {"branch": "main",    "suffix": "PRD"}, 
] 

# Admins to add to workspaces and connections, with format:
# Allowed User types: "User", "Group", "ServicePrincipal", "ServicePrincipalProfile"
# See pf.get_service_principal_id and pf.get_user_id to retrieve the user_uuid for service principals and users, respectively.
roles = [
    {
        "user_uuid": "8ff12cf8-7085-457e-a5b2-566c96a74076",   
        "user_type": "User"
    },
    {
        "user_uuid": "56b356c5-7f22-419e-b8b3-2fc6112fedae", 
        "user_type": "Group"
    }
]


# Authenticating method depends on where you are running the script.
# If you are running the script inside of Microsoft Fabric, make sure to fill the Key Vault. 
# Authenticate using "oauth" if in VS Code for authentication with interactive mode. 
# Authenticate using "env" for service principal authentication with environment variables FAB_TENANT_ID, FAB_CLIENT_ID, FAB_CLIENT_SECRET
# Locally, you can set these environment variables in your terminal or IDE. 
authentication_method = "env" # Options: "fabric", "env", "oauth"


# If you are running the script inside of Microsoft Fabric, make sure to fill the Key Vault.
key_vault = "https://jaircampelo-kv.vault.azure.net/"



In [ ]:
# Authentication and Logging
import os
import pyfabricops as pf

pf.set_auth_provider("env") # Authenticate using "env" for service principal authentication with environment variables FAB_TENANT_ID, FAB_CLIENT_ID, FAB_CLIENT_SECRET

# Setup logger
pf.setup_logging("info", "detailed") 
logger = pf.get_logger(__name__)


In [ ]:
# Create Workspaces
workspaces = []
capacity_id = pf.resolve_capacity(capacity) 

for branch in branches:
    branch_suffix = branch["suffix"]
    
    workspace_name = f"{project_name}-{branch_suffix}"
    
    workspace_created = pf.create_workspace(
        display_name=workspace_name,
        capacity=capacity_id,
        df=False,
    )
    
    # If workspace creation fails (e.g. because it already exists), retrieve the existing workspace
    if workspace_created:
        workspaces.append(workspace_created)
        workspace_id = workspace_created["id"]
        logger.success(f"Workspace {workspace_name} created with ID {workspace_id}")
    else:
        workspace_retrivied = pf.get_workspace(workspace_name, df=False)
        workspaces.append(workspace_retrivied)
        workspace_id = workspace_retrivied["id"]
        logger.warning(f"Workspace {workspace_name} already exists with ID {workspace_id}")

    # Assign admins to workspace
    for role in roles:
        pf.add_workspace_role_assignment(
            workspace_name, 
            role["user_uuid"], 
            role["user_type"], 
            role="Admin",
            df=False
        )

display(pf.json_to_df(workspaces)) 


In [ ]:
# Check capacity assign to workspace
import time    
    
MAX_RETRIES = 10
RETRY_INTERVAL = 10
logger.info(f"Checking capacity assign progress...")

for attempt in range(1, MAX_RETRIES + 1):
    logger.info(f"Attempt {attempt}/{MAX_RETRIES}")
    workspace = pf.get_workspace(f"{project_name}-PRD", df=False)
    
    if workspace["capacityAssignmentProgress"] == "Completed":
        logger.success(f"Capacity assigned successfully.")
        break

    else: 
        if attempt < MAX_RETRIES:
            time.sleep(RETRY_INTERVAL)
        else:
            logger.error(f"Capacity assignment failed after {MAX_RETRIES} attempts.")
            raise TimeoutError(f"Capacity assignment did not complete within {MAX_RETRIES * RETRY_INTERVAL} seconds.") 
              

In [ ]:
# Create Lakehouses, Pools and Environments in each Workspace
for workspace in workspaces:

    # Create folders in workspace
    folders = ["data", "utils", "notebooks", "pipelines", "reporting"] 
    for folder in folders:
        pf.create_folder(
            workspace=workspace["id"],
            display_name=folder,
            df=False,
        )

    # Create Lakehouse in workspace
    pf.create_lakehouse(
        workspace=workspace["id"],
        display_name=f"lh_{project_name}",
        folder="data",
        enable_schemas=True,
        df=False,
    )

    # Create Spark pool in workspace
    # Adjust node_family, node_size, and max_node_count based on your needs and budget. 
    # See documentation for details: https://learn.microsoft.com/en-us/fabric/data-engineering/spark-compute
    pool_created = pf.create_workspace_custom_pool(
        workspace=workspace["id"],
        display_name="default_pool",
        auto_scale_enabled=True,
        min_node_count=1,
        max_node_count=16,
        dynamic_executor_allocation_enabled=True,
        min_executors=1,
        max_executors=15,
        node_family="MemoryOptimized",
        node_size="Medium", # Options: Small, Medium, Large
        df=False,
    )

    # If pool creation fails (e.g. because it already exists), retrieve the existing pool
    if pool_created is None:
        pool_created = pf.get_workspace_custom_pool(
            workspace=workspace["id"],
            workspace_custom_pool="default_pool",
            df=False,
        )

    # Create Spark environment in workspace and link to pool
    environment_created = pf.create_environment(
        workspace=workspace["id"],
        display_name="env_default",
        folder = "utils",
        df=False,
    )

    # If environment creation fails (e.g. because it already exists), retrieve the existing environment
    if environment_created is None:
        environment_created = pf.get_environment(
            workspace=workspace["id"],
            environment="env_default",
            df=False,
        )

    # Update environment with Spark compute settings and link to pool
    pf.update_environment_spark_compute(
        workspace=workspace["id"],
        environment="env_default",
        pool="default_pool",
        driver_cores=8,
        driver_memory="56g",
        executor_cores=8,
        executor_memory="56g",
        dynamic_executor_allocation_enabled=True,
        min_executors=1,
        max_executors=15,
        spark_properties=[
            {"key": "spark.sql.caseSensitive", "value": True},
            {"key": "spark.native.enabled", "value": True},
        ],
        runtime_version="1.3"
    )

    # Prepare environment by installing external libraries and publish environment for use in workspace.
    pf.delete_path("../tmp") 

    # Publish environment for use in workspace.
    pf.publish_environment(
        workspace=workspace["id"],
        environment=environment_created["id"],
        df=False
    )

    # Update workspace Spark settings and link to pool and environment. 
    # This will ensure that users can select the environment and pool when creating notebooks and pipelines, 
    # and that Spark settings are applied by default.
    pf.update_workspace_spark_settings(
        workspace=workspace["id"],
        automatic_log_enabled=True,
        high_concurrency_notebook_interactive_run_enabled=True,
        high_concurrency_notebook_pipeline_run_enabled=True,
        pool_customize_compute_enabled=True,
        pool_default_name="default_pool",
        pool_default_id=pool_created["id"],
        pool_default_type="Workspace",
        starter_pool_max_node_count= 1,
        starter_pool_max_executors=10,
        environment_name="env_default",
        environment_runtime_version="1.3",
        job_conservative_job_admission_enabled=True,
        job_session_timeout_in_minutes=20,
        df=False
    )

    logger.success(f'Pools and environments were created successfully in Workspace: {workspace["displayName"]}')


In [ ]:
# Create Variable Library Definition Files for workspaces
pf.delete_path("../tmp")

for b in branches:
    suffix = b["suffix"]
    branch = b["branch"]

    workspace = pf.get_workspace(f"{project_name}-{suffix}", df=False)
    
    pool = pf.get_workspace_custom_pool(
        workspace=workspace["id"],
        workspace_custom_pool="default_pool",
        df=False,
    )

    lakehouse = pf.get_lakehouse(workspace=workspace["id"], lakehouse=f"lh_{project_name}", df=False)
    lakehouse_id = lakehouse["id"]
    lakehouse_sql_endpoint_connection_string = lakehouse["properties"]["sqlEndpointProperties"]["connectionString"]
    lakehouse_sql_endpoint_id = lakehouse["properties"]["sqlEndpointProperties"]["id"]

    value_set = {
      "$schema": "https://developer.microsoft.com/json-schemas/fabric/item/variableLibrary/definition/valueSet/1.0.0/schema.json",
      "name": branch,
      "variableOverrides": [
        {
          "name": "workspace_name",
          "value": workspace["displayName"]
        },
        {
          "name": "workspace_id",
          "value": workspace["id"]
        },
        {
          "name": "pool_id",
          "value": pool["id"]
        },
        {
          "name": "lakehouse_id",
          "value": lakehouse_id
        },
        {
          "name": "lakehouse_sql_endpoint_connection_string",
          "value": lakehouse_sql_endpoint_connection_string
        },
        {
          "name": "lakehouse_sql_endpoint_id",
          "value": lakehouse_sql_endpoint_id
        },
      ]
    }

    pf.write_json(
        value_set,
        f"../tmp/vl_variables.VariableLibrary/valueSets/{branch}.json"
    )
   
pf.write_json(
    {
      "$schema": "https://developer.microsoft.com/json-schemas/fabric/gitIntegration/platformProperties/2.0.0/schema.json",
      "metadata": {
        "type": "VariableLibrary",
        "displayName": "vl_variables",
        "description": ""
      },
      "config": {
        "version": "2.0",
        "logicalId": "00000000-0000-0000-0000-000000000000",
      }
    },
    "../tmp/vl_variables.VariableLibrary/.platform"
)

pf.write_json(
    {
      "$schema": "https://developer.microsoft.com/json-schemas/fabric/item/variableLibrary/definition/settings/1.0.0/schema.json",
      "valueSetsOrder": [
        "develop",
        "main"
      ]
    },
    "../tmp/vl_variables.VariableLibrary/settings.json"
)

pf.write_json(
    {
      "$schema": "https://developer.microsoft.com/json-schemas/fabric/item/variableLibrary/definition/variables/1.0.0/schema.json",
      "variables": [
        {
          "name": "workspace_id",
          "note": "",
          "type": "String",
          "value": ""
        },
        {
          "name": "workspace_name",
          "note": "",
          "type": "String",
          "value": ""
        },
        {
          "name": "pool_id",
          "note": "",
          "type": "String",
          "value": ""
        },
        {
          "name": "lakehouse_id",
          "note": "",
          "type": "String",
          "value": ""
        },
        {
          "name": "lakehouse_sql_endpoint_connection_string",
          "note": "",
          "type": "String",
          "value": ""
        },
        {
          "name": "lakehouse_sql_endpoint_id",
          "note": "",
          "type": "String",
          "value": ""
        },
      ]
    },
    "../tmp/vl_variables.VariableLibrary/variables.json"
)

logger.success(f"Created the temporary Variable Library template in ../tmp/vl_variables.VariableLibrary")

In [ ]:
# Create Variable Libraries in each Workspace
for b in branches:
    suffix = b["suffix"]
    pf.create_variable_library(
        workspace=f"{project_name}-{suffix}",
        display_name="vl_variables",
        item_definition=pf.pack_item_definition(path="../tmp/vl_variables.VariableLibrary"),
        folder="utils",
    ) 
    logger.success(f"Variable Library vl_libraries created in workspace {project_name}-{suffix}")


In [ ]:
connection_name = f'conn_github_{project_name}'
workspace_id = pf.resolve_workspace(f'{project_name}-DEV')
repository_name = 'end-to-end-healthy-analytics'
repository_url = f'https://github.com/jaircampelo/{repository_name}'

github_connection = pf.create_github_source_control_connection(
    display_name=connection_name,
    repository=repository_url,
    github_token=os.getenv('GH_TOKEN'),
    df=False,
)

# If connection fail (e.g. beacause it already exists), retrieve the existing connection
if github_connection is not None:
    display(github_connection)

connection_id = pf.resolve_connection(connection_name)

# Assing owners and users to connection
for role in roles:
    if role['user_type'] == 'Group':
        pf.add_connection_role_assignment(
            connection_name,
            role['user_uuid'],
            role['user_type'],
            role='User',
            df=False,
        )
    else:
        pf.add_connection_role_assignment(
        connection_name,
        role['user_uuid'],
        role['user_type'],
        role='Owner',
        df=False,
    )
        
# Connect Github repository to DEV workspace
pf.github_connect(
    workspace=workspace_id,
    connection=connection_id,
    owner_name='jaircampelo',
    repository_name=repository_name,
    branch_name='develop',
    directory_name='src',
)

# If the repository was already connected, update the connection settings to ensure they are correct
pf.update_my_git_connection(
    workspace=workspace_id,
    request_body_type='UpdateGitCredentialsToConfiguredConnectionRequest',
    connection_id=connection_id,
)

# Innitialize Git repository in workspace and make initial commit
pf.git_init(workspace=workspace_id, initialize_strategy='PreferWorkspace', df=False)
pf.commit_to_git(workspace=workspace_id, mode='All', comment='Initial Fabric Commit', df=False)